In [0]:
%pip install azure-identity azure-storage-file-datalake

In [0]:
dbutils.library.restartPython()

In [0]:
%run "../config/00_config"

In [0]:
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

storage_account_name = ADLS_STORAGE_ACCOUNT_NAME
container_name = ADLS_CONTAINER

credential = ClientSecretCredential(
    tenant_id=ADLS_TENANT_ID,
    client_id=ADLS_CLIENT_ID,
    client_secret=ADLS_CLIENT_SECRET
)

service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=credential
)

file_system_client = service_client.get_file_system_client(
    file_system=container_name
)

print("Conexão criada com sucesso.")
print(f"Storage Account: {storage_account_name}")
print(f"Container: {container_name}")

In [0]:
paths = file_system_client.get_paths(path="")

dados = []

for path in paths:
    dados.append({
        "nome": path.name,
        "tipo": "pasta" if path.is_directory else "arquivo",
        "tamanho_bytes": path.content_length,
        "ultima_modificacao": path.last_modified
    })

df_tudo = spark.createDataFrame(dados)

display(df_tudo)

In [0]:
from pyspark.sql.functions import col, split, size

df_primeiro_nivel = (
    df_tudo
    .withColumn("partes", split(col("nome"), "/"))
    .filter(size(col("partes")) == 1)
    .drop("partes")
)

display(df_primeiro_nivel)

In [0]:
from pyspark.sql.functions import col

display(
    df_tudo.filter(
        col("nome").endswith(".csv")
    )
)

In [0]:
from pyspark.sql.functions import col

display(
    df_tudo.filter(
        col("nome").contains("ecommerce")
    )
)